# Optimizer Geometry — Part 2 (remaining runs only)

The full SGD sweep and Adam at batch 32 are **already done** (the salvaged results).
This notebook runs *only what's left*: the rest of the **Adam sweep** (batch 64–1024,
3 seeds) plus one small-batch **SGD** run, and it saves the checkpoints needed for the
**loss-interpolation** and **preconditioned-curvature (Cohen)** plots.

**No Google Drive.** It downloads results straight to your Mac: a backup zip every few
runs (so a dropped connection can't lose finished runs) and a final zip at the end.

### How to run
1. `Runtime → Change runtime type → T4 GPU → Save`, then `Runtime → Run all`.
2. **Before running, in a Terminal run `caffeinate -dim` and leave it open** — this stops
   your Mac sleeping and dropping the Colab connection (that's what killed the first run).
   Keep the Colab tab open too.
3. When Chrome asks *"Allow multiple downloads?"*, click **Allow**.
4. When it finishes, send me the **`..._FINAL.zip`** (or, if it disconnects, the most
   recent `..._backup_runNN.zip`). The printed log is also a safety net.

Roughly a couple of hours on a free GPU — but treat that as a rough figure, it varies.

**Want to test it first?** Set `TEST_MODE = True` in the config cell and Run all: it runs a
~1-minute synthetic version **on CPU (no GPU needed)** so you can confirm it works *now*,
even while the GPU quota is still locked. Then set `TEST_MODE = False` and switch to GPU.

In [ ]:
# ===================== CONFIG =====================
CONFIG = dict(
    dataset="FashionMNIST",
    epochs=30,
    sgd_lr=0.05, sgd_momentum=0.9,
    adam_lr=1e-3,
    weight_decay=0.0,
    geom_subset=1024,
    hvp_iters=20,
    sharp_dirs=10, sharp_rho=0.01,
    interp_points=21,
    out_dir="results_geometry_part2",
)

# The remaining runs. SGD bs=32 first (flat interpolation endpoint), then the full
# single-seed Adam sweep (so even a partial run is useful), then the other seeds.
ADAM_BS = [64, 128, 256, 512, 1024]
RUN_SPECS = (
    [("sgd", 32, 0)]
    + [("adam", bs, 0) for bs in ADAM_BS]
    + [("adam", bs, s) for s in [1, 2] for bs in ADAM_BS]
)  # -> 1 + 5 + 10 = 16 runs

DOWNLOAD_EVERY = 3   # download a cumulative backup zip every N runs

# Quick smoke test (CPU, ~1 min, no GPU): set True, Run all, then set back to False.
TEST_MODE = False
if TEST_MODE:
    CONFIG = {**CONFIG, "dataset": "synthetic", "epochs": 1, "out_dir": "results_part2_test"}
    RUN_SPECS = [("sgd", 32, 0), ("adam", 64, 0), ("adam", 1024, 0)]
    DOWNLOAD_EVERY = 99

In [ ]:
import os, json, time, random, math, copy, csv, shutil
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import grad
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# Detect Colab so downloads work there and are skipped elsewhere (e.g. a local test).
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
DATASET_STATS = {
    "FashionMNIST": dict(ch=1, mean=(0.2860,), std=(0.3530,)),
    "CIFAR10": dict(ch=3, mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)),
}


def get_data(name):
    if name == "synthetic":
        Xtr = torch.randn(512, 1, 28, 28); ytr = torch.randint(0, 10, (512,))
        Xte = torch.randn(256, 1, 28, 28); yte = torch.randint(0, 10, (256,))
        return TensorDataset(Xtr, ytr), TensorDataset(Xte, yte), 1
    from torchvision import datasets, transforms
    st = DATASET_STATS[name]
    tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(st["mean"], st["std"])])
    cls = {"FashionMNIST": datasets.FashionMNIST, "CIFAR10": datasets.CIFAR10}[name]
    tr = cls("./data", train=True, download=True, transform=tf)
    te = cls("./data", train=False, download=True, transform=tf)
    return tr, te, st["ch"]


def geom_batch(train_set, n, device):
    idx = list(range(min(n, len(train_set))))
    xs = torch.stack([train_set[i][0] for i in idx])
    ys = torch.tensor([int(train_set[i][1]) for i in idx])
    return xs.to(device), ys.to(device)


class SmallCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 4 * 4, 128), nn.ReLU(), nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

In [ ]:
def tparams(model):
    return [p for p in model.parameters() if p.requires_grad]


def _raw_hvp(model, crit, x, y, vec, params):
    g = grad(crit(model(x), y), params, create_graph=True)
    flat_g = torch.cat([gi.reshape(-1) for gi in g])
    hv = grad((flat_g * vec).sum(), params, retain_graph=False)
    return torch.cat([h.reshape(-1) for h in hv]).detach()


def top_hessian_eigenvalue(model, crit, x, y, iters=20):
    params = tparams(model)
    v = torch.randn(sum(p.numel() for p in params), device=x.device); v /= v.norm()
    for _ in range(iters):
        Hv = _raw_hvp(model, crit, x, y, v, params)
        nv = Hv.norm()
        if nv == 0:
            break
        v = Hv / nv
    return float((v * _raw_hvp(model, crit, x, y, v, params)).sum().item())


def random_direction_sharpness(model, crit, x, y, rho=0.01, n_dirs=10):
    params = tparams(model)
    w0 = parameters_to_vector(params).detach().clone()
    wn = w0.norm()
    with torch.no_grad():
        base = crit(model(x), y).item()
    worst = 0.0
    for _ in range(n_dirs):
        d = torch.randn_like(w0); d /= d.norm()
        vector_to_parameters(w0 + rho * wn * d, params)
        with torch.no_grad():
            worst = max(worst, crit(model(x), y).item() - base)
    vector_to_parameters(w0, params)
    return worst

In [ ]:
@torch.no_grad()
def evaluate(model, loader, crit):
    model.eval(); tl = tc = tn = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        o = model(x); l = crit(o, y)
        tl += l.item() * y.size(0); tc += (o.argmax(1) == y).sum().item(); tn += y.size(0)
    return tl / tn, tc / tn


def train_model_fast(opt_name, bs, seed, train_set, test_set, in_ch, cfg):
    """Same training as before, but we read train loss/acc from the training pass itself
    instead of doing a second full pass over the training set each epoch. The model trains
    identically; this just drops a redundant evaluation, which is the main speed-up."""
    set_seed(seed)
    model = SmallCNN(in_ch).to(DEVICE)
    crit = nn.CrossEntropyLoss()
    if opt_name == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=cfg["sgd_lr"],
                              momentum=cfg["sgd_momentum"], weight_decay=cfg["weight_decay"])
    else:
        opt = torch.optim.Adam(model.parameters(), lr=cfg["adam_lr"], weight_decay=cfg["weight_decay"])
    nw = 2 if DEVICE.type == "cuda" else 0
    pm = DEVICE.type == "cuda"
    tr_loader = DataLoader(train_set, batch_size=bs, shuffle=True, num_workers=nw, pin_memory=pm)
    te_loader = DataLoader(test_set, batch_size=512, shuffle=False, num_workers=nw, pin_memory=pm)
    best_acc, best_state, history = -1.0, None, []
    for ep in range(1, cfg["epochs"] + 1):
        model.train()
        run_loss = run_correct = run_n = 0
        for x, y in tr_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            out = model(x)
            loss = crit(out, y)
            loss.backward()
            opt.step()
            with torch.no_grad():
                run_loss += loss.item() * y.size(0)
                run_correct += (out.argmax(1) == y).sum().item()
                run_n += y.size(0)
        tel, tea = evaluate(model, te_loader, crit)  # test eval each epoch (cheap; for best + curve)
        history.append(dict(epoch=ep, train_loss=run_loss / run_n, train_acc=run_correct / run_n,
                            test_loss=tel, test_acc=tea))
        if tea > best_acc:
            best_acc, best_state = tea, copy.deepcopy(model.state_dict())
    return model, opt, history, best_acc, history[-1]["test_acc"], best_state

In [ ]:
def save_csv(results, out):
    with open(out / "results.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        w.writeheader(); w.writerows(results)


def run_interpolation(ref, gx, gy, in_ch, cfg):
    if "sgd_small" not in ref or "adam_large" not in ref:
        print("Interpolation endpoints missing; skipping (can be done later from checkpoints).")
        return None
    model = SmallCNN(in_ch).to(DEVICE); crit = nn.CrossEntropyLoss()
    w1, w2 = ref["sgd_small"], ref["adam_large"]
    alphas = np.linspace(-0.5, 1.5, cfg["interp_points"])
    params = tparams(model); losses = []
    with torch.no_grad():
        for a in alphas:
            vector_to_parameters((1 - a) * w1 + a * w2, params)
            losses.append(crit(model(gx), gy).item())
    return alphas, losses

In [ ]:
def maybe_download(path):
    if IN_COLAB:
        try:
            files.download(str(path))
        except Exception as e:
            print("   (download failed:", e, "- grab it from the file browser)")


def run_second_part(cfg, specs, download_every=3):
    out = Path(cfg["out_dir"])
    (out / "figures").mkdir(parents=True, exist_ok=True)
    (out / "checkpoints").mkdir(parents=True, exist_ok=True)
    print("Device:", DEVICE, "| dataset:", cfg["dataset"], "| on Colab:", IN_COLAB,
          "| runs to do:", len(specs))
    train_set, test_set, in_ch = get_data(cfg["dataset"])
    gx, gy = geom_batch(train_set, cfg["geom_subset"], DEVICE)
    crit = nn.CrossEntropyLoss()
    results, ref, t0 = [], {}, time.time()
    for i, (o, bs, s) in enumerate(specs, 1):
        model, opt, hist, best_acc, final_acc, best_state = train_model_fast(
            o, bs, s, train_set, test_set, in_ch, cfg)
        top = top_hessian_eigenvalue(model, crit, gx, gy, cfg["hvp_iters"])
        sharp = random_direction_sharpness(model, crit, gx, gy, cfg["sharp_rho"], cfg["sharp_dirs"])
        row = dict(dataset=cfg["dataset"], optimizer=o, batch_size=bs, seed=s,
                   final_test_acc=final_acc, best_test_acc=best_acc,
                   final_train_loss=hist[-1]["train_loss"], final_test_loss=hist[-1]["test_loss"],
                   top_hessian=top, sharpness=sharp)
        results.append(row)
        # Save full artifacts incl. optimizer state -> enables the preconditioned-curvature
        # (Cohen) analysis later without re-training.
        torch.save(dict(model_final=model.state_dict(), model_best=best_state,
                        optim_state=opt.state_dict(), history=hist, row=row),
                   out / "checkpoints" / f"{cfg['dataset']}_{o}_bs{bs}_seed{s}.pt")
        if o == "sgd" and bs == 32 and s == 0:
            ref["sgd_small"] = parameters_to_vector(tparams(model)).detach().clone()
        if o == "adam" and bs == 1024 and s == 0:
            ref["adam_large"] = parameters_to_vector(tparams(model)).detach().clone()
        json.dump(results, open(out / "results.json", "w"), indent=2)
        save_csv(results, out)
        print(f"[{i:2d}/{len(specs)}] {o:4s} bs={bs:<4d} seed={s} | "
              f"test_acc={final_acc:.4f}  top_H={top:8.3f}  sharp={sharp:.5f} | "
              f"{time.time() - t0:.0f}s elapsed")
        if i % download_every == 0:
            zp = shutil.make_archive(str(out) + f"_backup_run{i:02d}", "zip", str(out))
            print(f"   -> backup zip after run {i}; downloading…")
            maybe_download(zp)
    # Loss interpolation between the flat (SGD bs=32) and sharp (Adam bs=1024) minima.
    interp = run_interpolation(ref, gx, gy, in_ch, cfg)
    if interp is not None:
        a, l = interp
        plt.figure(); plt.plot(a, l, marker="o")
        plt.axvline(0, ls="--", c="gray"); plt.axvline(1, ls="--", c="gray")
        plt.xlabel("alpha  (0 = SGD bs=32, 1 = Adam bs=1024)"); plt.ylabel("train loss")
        plt.title("Loss interpolation: flat (SGD small-batch) vs sharp (Adam large-batch)")
        plt.grid(alpha=.3)
        plt.savefig(out / "figures" / "interpolation.png", dpi=140, bbox_inches="tight"); plt.show()
        json.dump(dict(alphas=[float(x) for x in a], losses=[float(x) for x in l]),
                  open(out / "interpolation.json", "w"), indent=2)
        print("Saved interpolation figure + data.")
    final_zip = shutil.make_archive(str(out) + "_FINAL", "zip", str(out))
    print("\nFINAL zip ready ->", final_zip)
    maybe_download(final_zip)
    print("Done. Send me the FINAL zip (or the most recent backup zip if it disconnected).")
    return results

In [ ]:
results = run_second_part(CONFIG, RUN_SPECS, download_every=DOWNLOAD_EVERY)